### 1. Run Tomofast Forward Modelling

In [ ]:
# =============================================================================
# VALIDATE EQUATION 10 (Bruce et al. 2025) AGAINST ACTUAL TOMOFAST-X RUNS
# =============================================================================
# Scans every Tomofast_Output/<body>/log.txt, pulls out:
#     COMPRESSION RATE  =  <value>
#     COMPRESSION ERROR, r =  <value>
# and plots compression error (log y-axis) vs compression rate, with a red
# dashed line at the 1% (0.01) target so you can see which runs landed above
# or below the threshold the equation was supposed to hit.
# =============================================================================

import re
import os
from pathlib import Path
import matplotlib.pyplot as plt

# ============================================================
#  USER CONFIG -- adjust to match the folder layout
# ============================================================
base_dir   = Path(os.getcwd())
output_dir = base_dir / "Tec_files_bellon" / "converted" / "Tomofast_Output"
TARGET_ERROR = 0.01   # 1% target from Bruce et al.
WAVELET = "haar"       # must match what calc_ideal_compression_rate() used when generating the parfile

# Equation 10 constants (Table 2, Bruce et al. 2025)
EQ10_CONSTANTS = {
    "haar": dict(a=35.07, k=-0.872, b=-0.884),
    "d4":   dict(a=42.53, k=-0.614, b=-0.852),
}

def predicted_error_for_rate(rate, n_model, wavelet=WAVELET):
    """
    Inverts Equation 10 (c_r = a * c_e^k * N^b) to ask: given the rate that
    was actually used, what compression error did the equation expect?
        c_e = (c_r / (a * N^b)) ** (1/k)
    """
    c = EQ10_CONSTANTS[wavelet.lower()]
    return (rate / (c["a"] * (n_model ** c["b"]))) ** (1.0 / c["k"])

# ============================================================
#  PARSE LOG FILES
# ============================================================
rate_re   = re.compile(r"COMPRESSION RATE\s*=\s*([0-9.eE+\-]+)")
error_re  = re.compile(r"COMPRESSION ERROR,\s*r\s*=\s*([0-9.eE+\-]+)")
memory_re = re.compile(r"MEMORY USED \(total\)\s*\[GB\]\s*=\s*([0-9.eE+\-]+)")
size_re   = re.compile(r"modelGrid\.size\s*=\s*([0-9]+)\s+([0-9]+)\s+([0-9]+)")

results = []  # list of dicts: body, rate, error, memory_gb, n_model, predicted_error

log_files = sorted(output_dir.glob("*/log.txt"))
print(f"Found {len(log_files)} log.txt file(s) under {output_dir}\n")

for log_path in log_files:
    body = log_path.parent.name
    text = log_path.read_text(errors="ignore")

    rate_match   = rate_re.search(text)
    error_match  = error_re.search(text)
    memory_match = memory_re.search(text)
    size_match   = size_re.search(text)

    if not rate_match or not error_match:
        print(f"  [skip] {body}: could not find rate/error in log")
        continue

    rate   = float(rate_match.group(1))
    error  = float(error_match.group(1))
    memory = float(memory_match.group(1)) if memory_match else None

    n_model = None
    predicted_error = None
    if size_match:
        nx, ny, nz = (int(size_match.group(i)) for i in (1, 2, 3))
        n_model = nx * ny * nz
        predicted_error = predicted_error_for_rate(rate, n_model)

    results.append(dict(body=body, rate=rate, error=error, memory_gb=memory,
                         n_model=n_model, predicted_error=predicted_error))

    mem_str  = f"{memory:.4f} GB" if memory is not None else "n/a"
    pred_str = f"{predicted_error:.5e}" if predicted_error is not None else "n/a"
    print(f"  {body:40s}  rate={rate:.5f}  error={error:.5e}  "
          f"predicted_error={pred_str}  memory={mem_str}")

if not results:
    raise SystemExit("No valid (rate, error) pairs found -- check output_dir / regex patterns.")

# ============================================================
#  PLOT: compression error (log scale) vs compression rate
# ============================================================
rates  = [r["rate"] for r in results]
errors = [r["error"] for r in results]
bodies = [r["body"] for r in results]

fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(rates, errors, color="tab:blue", zorder=3)

for r, e, b in zip(rates, errors, bodies):
    ax.annotate(b, (r, e), fontsize=7, alpha=0.7,
                xytext=(4, 4), textcoords="offset points")

ax.axhline(TARGET_ERROR, color="red", linestyle="--", linewidth=1.5,
           label=f"target error = {TARGET_ERROR:.0%}")

ax.set_yscale("log")
ax.set_xlabel("Compression rate, $c_r$")
ax.set_ylabel("Compression error, $c_e$ (log scale)")
ax.set_title("Validation of Equation 10 (Bruce et al. 2025)\nActual compression error vs rate, per body")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
out_path = output_dir / "compression_validation.png"
fig.savefig(out_path, dpi=150)
print(f"\nSaved plot: {out_path}")

# ============================================================
#  PLOT 2: memory used (GB) per body -- check if memory stayed
#          consistent across bodies/runs, or scales as expected
# ============================================================
mem_results = [r for r in results if r["memory_gb"] is not None]

if mem_results:
    mem_bodies = [r["body"] for r in mem_results]
    mem_values = [r["memory_gb"] for r in mem_results]

    fig2, ax2 = plt.subplots(figsize=(8, 6))
    ax2.bar(range(len(mem_bodies)), mem_values, color="tab:green", zorder=3)
    ax2.set_xticks(range(len(mem_bodies)))
    ax2.set_xticklabels(mem_bodies, rotation=45, ha="right", fontsize=8)
    ax2.set_ylabel("Memory used (total) [GB]")
    ax2.set_title("Memory usage per body (from log.txt)")
    ax2.grid(True, axis="y", alpha=0.3)

    avg_mem = sum(mem_values) / len(mem_values)
    ax2.axhline(avg_mem, color="red", linestyle="--", linewidth=1.2,
                label=f"mean = {avg_mem:.4f} GB")
    ax2.legend()

    fig2.tight_layout()
    out_path2 = output_dir / "memory_validation.png"
    fig2.savefig(out_path2, dpi=150)
    print(f"Saved plot: {out_path2}")
else:
    print("\n[skip] No memory values found in any log -- check memory_re pattern.")

# ============================================================
#  PLOT 3: predicted error (Eq. 10, inverted) vs actual error
#          per body -- the real validation of the equation
# ============================================================
eq_results = [r for r in results if r["predicted_error"] is not None]

if eq_results:
    eq_bodies    = [r["body"] for r in eq_results]
    eq_actual    = [r["error"] for r in eq_results]
    eq_predicted = [r["predicted_error"] for r in eq_results]

    x = range(len(eq_bodies))
    width = 0.35

    fig3, ax3 = plt.subplots(figsize=(9, 6))
    ax3.bar([i - width/2 for i in x], eq_predicted, width,
            label="predicted (Eq. 10, inverted)", color="tab:orange", zorder=3)
    ax3.bar([i + width/2 for i in x], eq_actual, width,
            label="actual (Tomofast-x log)", color="tab:blue", zorder=3)

    ax3.axhline(TARGET_ERROR, color="red", linestyle="--", linewidth=1.2,
                label=f"target error = {TARGET_ERROR:.0%}")

    ax3.set_yscale("log")
    ax3.set_xticks(list(x))
    ax3.set_xticklabels(eq_bodies, rotation=45, ha="right", fontsize=8)
    ax3.set_ylabel("Compression error (log scale)")
    ax3.set_title(f"Equation 10 ({WAVELET}) predicted vs actual compression error, per body")
    ax3.legend()
    ax3.grid(True, which="both", axis="y", alpha=0.3)

    fig3.tight_layout()
    out_path3 = output_dir / "equation_validation.png"
    fig3.savefig(out_path3, dpi=150)
    print(f"Saved plot: {out_path3}")

    # console summary of the discrepancy
    print("\nPredicted vs actual error (ratio = actual / predicted):")
    for r in eq_results:
        ratio = r["error"] / r["predicted_error"]
        print(f"  {r['body']:40s}  n_model={r['n_model']:>9}  "
              f"predicted={r['predicted_error']:.4e}  actual={r['error']:.4e}  "
              f"ratio={ratio:.2f}x")
else:
    print("\n[skip] No modelGrid.size found in any log -- cannot compute predicted error "
          "(check size_re pattern, or the log doesn't include the parfile echo).")

plt.show()

In [ ]:
import os

base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output"
)

print("Renaming folders in:", tomofast_out)

for name in sorted(os.listdir(tomofast_out)):
    full_path = os.path.join(tomofast_out, name)
    if os.path.isdir(full_path) and name.startswith("TEC_"):
        new_name = name[len("TEC_"):]
        new_path = os.path.join(tomofast_out, new_name)
        print(f"Renaming: {name} -> {new_name}")
        os.rename(full_path, new_path)

print("Done.")

### 2. Plot Tomofast Output (Bxyz maps)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output"
)
print("Reading from:", tomofast_out)


def plot_bxyz(x_coords, y_coords, data, sfx, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ncols = data.shape[1]

    #  detect Bz-only vs full Bx,By,Bz data ----
    if ncols >= 6:
        # full format: x y z bx by bz
        components = [(3, "Bx", False), (4, "By", False), (5, "Bz", True)]
        bz_col = data[:, 5]
    elif ncols == 4:
        # Bz-only format: x y z bz
        components = [(3, "Bz", True)]
        bz_col = data[:, 3]
    else:
        print(f"[SKIP] {sfx}: unexpected number of columns ({ncols})")
        return

    scale = np.nanmax(np.abs(bz_col))
    n_panels = len(components)

    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    axes = np.atleast_1d(axes)  # makes single-panel case iterable too
    fig.suptitle(sfx, fontsize=14)

    for ax, (col, component, flip) in zip(axes, components):
        # flip only Bz from tomofast to qdm convention
        b_values = -data[:, col] if flip else data[:, col]
        df = pd.DataFrame({"x": x_coords, "y": y_coords, "b": b_values})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)

        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)

        if len(df) != nx * ny:
            print(f"[SKIP] {sfx} {component}: not a regular grid")
            continue

        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)

        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} (nT)", fontsize=12)
        cbar.ax.tick_params(labelsize=11)
        ax.set_xlabel("x (µm)", fontsize=12)
        ax.set_ylabel("y (µm)", fontsize=12)
        ax.set_title(component, fontsize=13)
        ax.tick_params(labelsize=11)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style='plain')
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}.pdf")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue
    filepath = os.path.join(sample_dir, "data", "mag_prior.txt")
    if not os.path.isfile(filepath):
        print("[missing]", filepath)
        continue
    data = np.loadtxt(filepath, skiprows=1)
    x_coords = data[:, 0]
    y_coords = data[:, 1]
    plot_bxyz(x_coords, y_coords, data, sfx, sample_dir)

print("\nDone.")

In [ ]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42   # embed as TrueType (Type 42), not Type 3
matplotlib.rcParams['ps.fonttype']  = 42   # same for EPS, if you ever use it

import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output"
)
print("Reading from:", tomofast_out)

# ============================================================
#  USER CONFIG
# ============================================================
FLIP_BZ_SIGN   = True    # Tomofast (positive-down) -> QDM/physical (positive-up) convention
SHOW_COLORBAR  = True     # set False if you'll add one shared colorbar manually in Inkscape
SHOW_AXES      = True     # set False for a clean, borderless panel (axes/labels added in Inkscape instead)
CMAP           = "seismic"
FIGSIZE        = (5, 5)    # per-panel size, inches -- keep small/square for easy Inkscape assembly
DPI            = 300


def detect_bz_column(data):
    """
    Auto-detects the Bz column for either format:
      4 columns:  x y z bz
      6+ columns: x y z bx by bz
    Returns the column index of Bz, or None if the format is unrecognised.
    """
    ncols = data.shape[1]
    if ncols == 4:
        return 3          # x y z bz
    elif ncols >= 6:
        return 5          # x y z bx by bz
    else:
        return None


def plot_bz_only(x_coords, y_coords, data, sfx, out_dir):
    os.makedirs(out_dir, exist_ok=True)

    bz_col = detect_bz_column(data)
    if bz_col is None:
        print(f"[SKIP] {sfx}: unexpected number of columns ({data.shape[1]})")
        return

    b_values = -data[:, bz_col] if FLIP_BZ_SIGN else data[:, bz_col]

    df = pd.DataFrame({"x": x_coords, "y": y_coords, "b": b_values})
    df = df.sort_values(["y", "x"]).reset_index(drop=True)
    unique_x = np.sort(df["x"].unique())
    unique_y = np.sort(df["y"].unique())
    nx, ny = len(unique_x), len(unique_y)
    if len(df) != nx * ny:
        print(f"[SKIP] {sfx} Bz: not a regular grid")
        return

    X = df["x"].values.reshape(ny, nx)
    Y = df["y"].values.reshape(ny, nx)
    Z = df["b"].values.reshape(ny, nx)

    scale = np.nanmax(np.abs(Z))

    fig, ax = plt.subplots(figsize=FIGSIZE)
    im = ax.contourf(X, Y, Z, levels=100, cmap=CMAP, vmin=-scale, vmax=scale)

    if SHOW_COLORBAR:
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label("Bz (nT)", fontsize=12)
        cbar.ax.tick_params(labelsize=11)

    if SHOW_AXES:
        ax.set_xlabel("x (µm)", fontsize=12)
        ax.set_ylabel("y (µm)", fontsize=12)
        ax.tick_params(labelsize=11)
        ax.ticklabel_format(useOffset=False, style='plain')
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))
    else:
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    ax.set_aspect("equal")

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bz_{sfx}.pdf")
    plt.savefig(filename, dpi=DPI, bbox_inches="tight")
    plt.close()
    print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue

    filepath = os.path.join(sample_dir, "data", "mag_prior.txt")
    if not os.path.isfile(filepath):
        print("[missing]", filepath)
        continue

    data = np.loadtxt(filepath, skiprows=1)
    x_coords = data[:, 0]
    y_coords = data[:, 1]

    plot_bz_only(x_coords, y_coords, data, sfx, sample_dir)

print("\nDone.")

### 3- Calculate parameters of the true models

In [ ]:
# ------------------------------------------------------------
# Per-cell parameters from Tomofast Forward magnetization + mesh
# LOOP over all body_* folders -> ONE nonzero-only CSV per body
# ------------------------------------------------------------

import os
import glob
import numpy as np
import pandas as pd

base_dir = os.getcwd()
TOMOFAST_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")

UM_TO_M = 1e-6


# ============================================================
# Magnetization -> Dec, Inc, |M|
#
#   |M|   = sqrt(Mx^2 + My^2 + Mz^2)
#   Inc   = arccos(-Mz / |M|) - 90
#           (Mz negated: Tomofast's internal z-axis is positive-down,
#            micromag/QDM convention is positive-up)
#   Dec   = 90 - atan2(My, Mx)
# ============================================================
def dec_inc_norm(Mx, My, Mz):
    Mx, My, Mz = np.atleast_1d(Mx, My, Mz)
    norm = np.sqrt(Mx**2 + My**2 + Mz**2)
    Mz_used = -Mz
    safe_norm = np.where(norm > 0, norm, np.nan)
    theta = np.degrees(np.arccos(np.clip(Mz_used / safe_norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc, norm


# ============================================================
# Fisher (1953) 95% confidence cone around the mean direction
#
#   alpha95 = arccos[ 1 - ((N-R)/R) * ((1/0.05)^(1/(N-1)) - 1) ]
#
# N = number of unit vectors, R = resultant length of their sum
# ============================================================
def alpha95(N, R):
    if R <= 0 or N <= 1:
        return np.nan
    term = (1 / 0.05) ** (1 / (N - 1)) - 1
    val = 1 - ((N - R) / R) * term
    val = np.clip(val, -1, 1)  # guard against tiny floating-point overshoot
    return np.degrees(np.arccos(val))


body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT, d))
)
print("Found bodies:", body_dirs)

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder = os.path.join(TOMOFAST_ROOT, body)
    mag_matches = glob.glob(os.path.join(body_folder, "Magnetization_*.txt"))
    mesh_matches = glob.glob(os.path.join(body_folder, "Meshgrid_*.txt"))

    if not mag_matches or not mesh_matches:
        print("   Missing Magnetization or Meshgrid file - skipping.")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag = pd.read_csv(mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                       names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag):
        print(f"   Cell count mismatch ({len(mesh)} vs {len(mag)}) - skipping.")
        continue

    df = pd.concat([mesh, mag], axis=1)
    df.insert(0, "CellID", np.arange(1, len(df) + 1))
    print(f"   Loaded {len(df)} cells.")

    # ------------------------------------------------------------
    # Cell geometry and volume
    #   dx = X2-X1, dy = Y2-Y1, dz = Z2-Z1   (converted um -> m)
    #   V  = dx * dy * dz
    # ------------------------------------------------------------
    df["dx"] = (df["X2"] - df["X1"]) * UM_TO_M
    df["dy"] = (df["Y2"] - df["Y1"]) * UM_TO_M
    df["dz"] = (df["Z2"] - df["Z1"]) * UM_TO_M
    df["Volume"] = df["dx"] * df["dy"] * df["dz"]

    # ------------------------------------------------------------
    # Per-cell direction and magnitude (see dec_inc_norm above)
    # ------------------------------------------------------------
    df["Dec"], df["Inc"], df["Mag_norm"] = dec_inc_norm(df["Mx"], df["My"], df["Mz"])

    # ------------------------------------------------------------
    # Per-cell magnetic moment
    #   m = M * V        (scalar, A.m^2)
    #   m_vec = (Mx,My,Mz) * V   (vector components, A.m^2)
    # ------------------------------------------------------------
    df["Moment"] = df["Mag_norm"] * df["Volume"]
    df["mx"] = df["Mx"] * df["Volume"]
    df["my"] = df["My"] * df["Volume"]
    df["mz"] = df["Mz"] * df["Volume"]

    cols = ["CellID", "i", "j", "k", "X1", "X2", "Y1", "Y2", "Z1", "Z2",
            "dx", "dy", "dz", "Volume", "Mx", "My", "Mz",
            "mx", "my", "mz",
            "Mag_norm", "Dec", "Inc", "Moment"]
    nz_out = df.loc[df["Mag_norm"] > 0, cols].copy()

    n = len(nz_out)
    if n == 0:
        print("   No nonzero cells found - skipping save.")
        continue

    # ============================================================
    # Fisher mean direction (Fisher, 1953)
    #
    #   unit vector per cell:  u_i = (Mx,My,Mz)_i / |M_i|
    #   resultant vector:      (X,Y,Z) = sum(u_i)
    #   resultant length:      R = sqrt(X^2 + Y^2 + Z^2)   [0 <= R <= N]
    #   mean direction:        Dec/Inc of (X,Y,Z)
    #
    # Equal weight per cell, independent of cell magnitude - avoids the
    # few-strong-cells bias that a plain component average would have.
    # ============================================================
    norm_per_cell = np.sqrt(nz_out["Mx"]**2 + nz_out["My"]**2 + nz_out["Mz"]**2)
    ux = nz_out["Mx"] / norm_per_cell
    uy = nz_out["My"] / norm_per_cell
    uz = nz_out["Mz"] / norm_per_cell

    X, Y, Z = ux.sum(), uy.sum(), uz.sum()
    R = np.sqrt(X**2 + Y**2 + Z**2)
    fisher_dec, fisher_inc, _ = dec_inc_norm(X, Y, Z)
    fisher_alpha95 = alpha95(n, R)

    # ------------------------------------------------------------
    # Amplitude-weighted mean direction (for comparison with Fisher mean)
    #   Dec/Inc of the simple component average (Mx_mean, My_mean, Mz_mean)
    # ------------------------------------------------------------
    mean_dec, mean_inc, _ = dec_inc_norm(nz_out["Mx"].mean(), nz_out["My"].mean(), nz_out["Mz"].mean())

    # ============================================================
    # Net (bulk) moment - vector sum over all nonzero cells
    #
    #   m_net = sum(m_i)                       (components summed first)
    #   |m_net| = sqrt(mx^2 + my^2 + mz^2)
    #
    # Preserves directional cancellation between cells - this is the
    # physically meaningful bulk moment a magnetometer would record.
    # ============================================================
    true_moment = np.array([
        nz_out["mx"].sum(),
        nz_out["my"].sum(),
        nz_out["mz"].sum()
    ])

    summary = {
        "Mean_Mag":              nz_out["Mag_norm"].mean(),
        "Mean_Dec":              mean_dec[0],
        "Mean_Inc":              mean_inc[0],
        "Fisher_Dec":            fisher_dec[0],
        "Fisher_Inc":            fisher_inc[0],
        "Fisher_R":              R,
        "Fisher_R_over_N":       R / n,
        "Fisher_alpha95":        fisher_alpha95,
        "Mean_Moment":           nz_out["Moment"].mean(),
        "True_mx":               true_moment[0],
        "True_my":               true_moment[1],
        "True_mz":               true_moment[2],
        "True_moment_magnitude": np.linalg.norm(true_moment),
    }

    print(f"   N_nonzero_cells = {n}")
    for key, val in summary.items():
        print(f"   {key:22s}: {val:,.25f}")

    for key in summary:
        nz_out[key] = np.nan
    nz_out.loc[nz_out.index[0], list(summary.keys())] = list(summary.values())

    out_path = os.path.join(body_folder, f"{body}_nonzero.csv")
    nz_out.to_csv(out_path, index=False)
    print("   Saved:", out_path)

print("\nDONE.")

### 3- Plot 3d voxels

### 4- Add noise to the forward data

In [ ]:
# ------------------------------------------------------------
# Add QDM noise + background field to Tomofast mag_prior data,
# looping over each body (same body-loop style as the voxel/glyph plots).
# Uses qdm_noise() from qdm_noise_utils.py (Souza-Junior & Uieda, 2023).
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qdm_noise_utils import qdm_noise

# ------------------------------------------------------------
#   USER PARAMETERS
# ------------------------------------------------------------
STD = 50          # noise standard deviation (nT)
BACKGROUND = 0  # constant background field added on top (nT)

base_dir = os.getcwd()
TOMOFAST_OUTPUT = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output")

print("Current working directory:", base_dir)
print("Looking for Tomofast_Output at:", TOMOFAST_OUTPUT)
print("  -> exists:", os.path.isdir(TOMOFAST_OUTPUT))
if not os.path.isdir(TOMOFAST_OUTPUT):
    raise FileNotFoundError(
        f"TOMOFAST_OUTPUT does not exist: {TOMOFAST_OUTPUT}\n"
        "Check that your notebook's working directory (os.getcwd()) is the folder "
        "that CONTAINS 'Tec_files_bellon', or edit TOMOFAST_OUTPUT above to an absolute path."
    )


def plot_difference(x_coords, y_coords, data_clean, data_noisy, sfx, out_dir, ncols):
    """
    Plots (noisy - clean) on its OWN color scale. The main Bxyz plots use a
    color scale set by the full field amplitude, which is usually far larger
    than the injected noise+background - so the noise is invisible there even
    though it's genuinely present in the data. This isolates just the change.
    """
    os.makedirs(out_dir, exist_ok=True)
    field_cols = [3, 4, 5] if ncols >= 6 else [3]
    labels = ["Bx", "By", "Bz"] if ncols >= 6 else ["Bz"]

    fig, axes = plt.subplots(1, len(field_cols), figsize=(6 * len(field_cols), 6))
    if len(field_cols) == 1:
        axes = [axes]
    fig.suptitle(f"{sfx}: noise + background actually injected (noisy - clean)", fontsize=13)

    x_r = np.round(x_coords, 9)
    y_r = np.round(y_coords, 9)

    for ax, col, component in zip(axes, field_cols, labels):
        diff = data_noisy[:, col] - data_clean[:, col]
        df = pd.DataFrame({"x": x_r, "y": y_r, "b": diff})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP diff] {sfx} {component}: not a regular grid")
            continue
        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)
        scale = np.nanmax(np.abs(Z))  # scaled to the DIFFERENCE itself, not the full field
        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} diff (nT)", fontsize=12)
        ax.set_xlabel("x ($\\mu$m)", fontsize=12)
        ax.set_ylabel("y ($\\mu$m)", fontsize=12)
        ax.set_title(f"{component} (mean={Z.mean():.1f} nT)", fontsize=13)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}_diff.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


def plot_bxyz(x_coords, y_coords, data, sfx, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ncols = data.shape[1]
    field_cols = [3, 4, 5] if ncols >= 6 else [3]
    labels = ["Bx", "By", "Bz"] if ncols >= 6 else ["Bz"]

    fig, axes = plt.subplots(1, len(field_cols), figsize=(6 * len(field_cols), 6))
    if len(field_cols) == 1:
        axes = [axes]
    fig.suptitle(sfx, fontsize=14)

    bz_col = data[:, field_cols[-1]]
    scale = np.nanmax(np.abs(bz_col))

    for ax, col, component in zip(axes, field_cols, labels):
        b_values = -data[:, col] if component == "Bz" else data[:, col]  # flip Bz, tomofast->qdm convention
        df = pd.DataFrame({"x": np.round(x_coords, 9), "y": np.round(y_coords, 9), "b": b_values})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP] {sfx} {component}: not a regular grid")
            continue
        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)
        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} (nT)", fontsize=12)
        ax.set_xlabel("x (µm)", fontsize=12)
        ax.set_ylabel("y (µm)", fontsize=12)
        ax.set_title(component, fontsize=13)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


all_entries = sorted(os.listdir(TOMOFAST_OUTPUT))
print(f"\nFound {len(all_entries)} entries in Tomofast_Output:", all_entries)

n_processed = 0

for body in all_entries:
    body_folder = os.path.join(TOMOFAST_OUTPUT, body)
    if not os.path.isdir(body_folder):
        print(f"  [skip] {body}: not a directory")
        continue

    filepath = os.path.join(body_folder, "data", "mag_prior.txt")
    backup_path = os.path.join(body_folder, "data", "mag_prior_no_noise.txt")

    if not os.path.isfile(filepath) and not os.path.isfile(backup_path):
        print(f"  [missing] {body}: no file at {filepath}")
        continue

    print(f"\n=== {body} ===")
    n_processed += 1

    # always add noise starting from the clean backup, never from a possibly
    # already-noisy mag_prior.txt - so re-running this cell never compounds noise
    source_path = backup_path if os.path.isfile(backup_path) else filepath
    if not os.path.isfile(backup_path):
        with open(filepath) as f_in, open(backup_path, "w") as f_out:
            f_out.write(f_in.read())

    with open(source_path) as f:
        header_line = f.readline()
    data_clean = np.atleast_2d(np.loadtxt(source_path, skiprows=1))
    ncols = data_clean.shape[1]

    x_coords, y_coords = data_clean[:, 0], data_clean[:, 1]
    x_rounded = np.round(x_coords, 9)
    y_rounded = np.round(y_coords, 9)
    nx = len(np.unique(x_rounded))
    ny = len(np.unique(y_rounded))
    print(f"  Detected grid: nx={nx}, ny={ny}, nx*ny={nx*ny}, n_rows={len(data_clean)}")

    data_noisy = data_clean.copy()
    field_idx = [3, 4, 5] if ncols >= 6 else [3]  # Bx,By,Bz if vector format, else Bz only

    for idx in field_idx:
        df = pd.DataFrame({"x": x_rounded, "y": y_rounded, "v": data_clean[:, idx]})
        df_sorted = df.sort_values(["y", "x"])
        if len(df) != nx * ny:
            print(f"  [WARNING] column {idx}: not a regular grid - noise not added.")
            continue
        grid = df_sorted["v"].to_numpy().reshape(ny, nx)
        noise = qdm_noise(shape=(ny, nx), std_dev=STD)
        noisy_grid = grid + noise + BACKGROUND
        df_sorted = df_sorted.copy()
        df_sorted["v_noisy"] = noisy_grid.flatten()
        data_noisy[:, idx] = df_sorted.sort_index()["v_noisy"].to_numpy()

    # ---- overwrite mag_prior.txt with the noisy version ----
    with open(filepath, "w") as f:
        f.write(header_line if header_line.strip() else f"{data_noisy.shape[0]}\n")
        for row in data_noisy:
            f.write(" ".join(f"{v:.16e}" for v in row) + "\n")
    print(f"  Added noise (std={STD} nT) + background ({BACKGROUND} nT)")
    print("  Overwrote:", filepath)

    # ---- explicit proof the values actually changed ----
    for idx in field_idx:
        diff = data_noisy[:, idx] - data_clean[:, idx]
        name = {3: "Bx", 4: "By", 5: "Bz"}.get(idx, f"col{idx}")
        print(f"    {name}: mean diff = {diff.mean():.3f} nT (should be close to BACKGROUND={BACKGROUND}), "
              f"identical-to-clean = {np.allclose(diff, 0)}")

    # ---- plot clean vs noisy, to see the difference ----
    plot_bxyz(x_coords, y_coords, data_clean, f"{body}_no_noise", body_folder)
    plot_bxyz(x_coords, y_coords, data_noisy, f"{body}_with_noise", body_folder)
    plot_difference(x_coords, y_coords, data_clean, data_noisy, body, body_folder, ncols)

print(f"\nDone. Processed {n_processed} body folder(s).")
if n_processed == 0:
    print("!!! NOTHING WAS PROCESSED. Check the path/folder listing printed above. !!!")